# Tutorial: ItemCF Baseline for MIND Ranking

Audience:
- This notebook is Member C's item-based collaborative filtering baseline.

Prerequisites:
- The MIND data is available under `data/train` and `data/valid`.
- `pandas`, `numpy`, and `scikit-learn` are available in the environment.

Learning goals:
- Build user-item interactions from training clicks.
- Score each validation candidate by similarity to user history.
- Evaluate `AUC`, `MRR`, `nDCG@5`, and `nDCG@10`.


In [1]:
from __future__ import annotations

import math
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find project root from {start}. Expected a parent directory containing 'data'."
    )

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

TRAIN_BEHAVIORS_PATH = PROJECT_ROOT / "data/train/behaviors.tsv"
VALID_BEHAVIORS_PATH = PROJECT_ROOT / "data/valid/behaviors.tsv"
OUTPUT_DIR = NOTEBOOK_DIR / "itemcf_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [TRAIN_BEHAVIORS_PATH, VALID_BEHAVIORS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

print("Notebook directory:", NOTEBOOK_DIR.resolve())
print("Project root:", PROJECT_ROOT.resolve())
print("Train behaviors:", TRAIN_BEHAVIORS_PATH)
print("Valid behaviors:", VALID_BEHAVIORS_PATH)
print("Output directory:", OUTPUT_DIR.resolve())


Notebook directory: /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/round1_baselines
Project root: /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit
Train behaviors: /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/data/train/behaviors.tsv
Valid behaviors: /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/data/valid/behaviors.tsv
Output directory: /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/round1_baselines/itemcf_output


In [2]:
BEHAVIORS_COLUMNS = ["impression_id", "user_id", "time", "history", "impressions"]

train_df = pd.read_csv(TRAIN_BEHAVIORS_PATH, sep="\t", header=None, names=BEHAVIORS_COLUMNS, dtype=str)
valid_df = pd.read_csv(VALID_BEHAVIORS_PATH, sep="\t", header=None, names=BEHAVIORS_COLUMNS, dtype=str)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)


Train shape: (156965, 5)
Valid shape: (73152, 5)


In [3]:
def parse_history(history_str: str):
    if not isinstance(history_str, str) or not history_str.strip():
        return []
    return history_str.strip().split()

def parse_impressions(impressions_str: str):
    news_ids = []
    labels = []
    if not isinstance(impressions_str, str) or not impressions_str.strip():
        return news_ids, labels
    for token in impressions_str.strip().split():
        if "-" not in token:
            continue
        news_id, label = token.rsplit("-", 1)
        try:
            news_ids.append(news_id)
            labels.append(int(label))
        except ValueError:
            continue
    return news_ids, labels


In [4]:
user_clicked = defaultdict(set)

for _, row in train_df.iterrows():
    user_id = row["user_id"]
    for news_id in parse_history(row["history"]):
        user_clicked[user_id].add(news_id)
    cand_ids, cand_labels = parse_impressions(row["impressions"])
    for nid, y in zip(cand_ids, cand_labels):
        if y == 1:
            user_clicked[user_id].add(nid)

item_users = defaultdict(set)
for user_id, items in user_clicked.items():
    for item in items:
        item_users[item].add(user_id)

print("Users with interactions:", len(user_clicked))
print("Items with interactions:", len(item_users))


Users with interactions: 50000
Items with interactions: 39865


In [5]:
def jaccard_similarity(item_a: str, item_b: str) -> float:
    users_a = item_users.get(item_a)
    users_b = item_users.get(item_b)
    if not users_a or not users_b:
        return 0.0
    inter = len(users_a & users_b)
    if inter == 0:
        return 0.0
    union = len(users_a | users_b)
    return inter / union

def itemcf_score(candidate_id: str, history_items: list[str], top_k: int = 20) -> float:
    if not history_items:
        return 0.0
    sims = [jaccard_similarity(candidate_id, hist_item) for hist_item in history_items]
    sims = [s for s in sims if s > 0]
    if not sims:
        return 0.0
    sims.sort(reverse=True)
    return float(np.mean(sims[:top_k]))

all_labels = []
all_scores = []
flat_rows = []

for _, row in valid_df.iterrows():
    user_id = row["user_id"]
    history_items = parse_history(row["history"])
    if not history_items:
        history_items = list(user_clicked.get(user_id, []))

    cand_ids, cand_labels = parse_impressions(row["impressions"])
    cand_scores = [itemcf_score(nid, history_items, top_k=20) for nid in cand_ids]

    if cand_labels:
        all_labels.append(cand_labels)
        all_scores.append(cand_scores)

        for nid, y, s in zip(cand_ids, cand_labels, cand_scores):
            flat_rows.append(
                {
                    "impression_id": row["impression_id"],
                    "candidate_news_id": nid,
                    "label": int(y),
                    "score": float(s),
                }
            )


In [6]:
def mean_auc(group_labels, group_scores):
    aucs = []
    for labels, scores in zip(group_labels, group_scores):
        if len(set(labels)) < 2:
            continue
        aucs.append(roc_auc_score(labels, scores))
    return float(np.mean(aucs)) if aucs else 0.0

def mrr_score(labels, scores):
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    for rank, idx in enumerate(order, start=1):
        if labels[idx] == 1:
            return 1.0 / rank
    return 0.0

def mean_mrr(group_labels, group_scores):
    values = [mrr_score(labels, scores) for labels, scores in zip(group_labels, group_scores)]
    return float(np.mean(values)) if values else 0.0

def dcg_at_k(labels_sorted, k):
    dcg = 0.0
    for i in range(min(k, len(labels_sorted))):
        rel = labels_sorted[i]
        dcg += (2 ** rel - 1) / math.log2(i + 2)
    return dcg

def ndcg_at_k(labels, scores, k):
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    ranked_labels = [labels[i] for i in order]
    dcg = dcg_at_k(ranked_labels, k)
    idcg = dcg_at_k(sorted(labels, reverse=True), k)
    return 0.0 if idcg == 0 else dcg / idcg

def mean_ndcg(group_labels, group_scores, k):
    values = [ndcg_at_k(labels, scores, k) for labels, scores in zip(group_labels, group_scores)]
    return float(np.mean(values)) if values else 0.0

metrics = {
    "AUC": mean_auc(all_labels, all_scores),
    "MRR": mean_mrr(all_labels, all_scores),
    "nDCG@5": mean_ndcg(all_labels, all_scores, 5),
    "nDCG@10": mean_ndcg(all_labels, all_scores, 10),
}
metrics


{'AUC': 0.5211644419870771,
 'MRR': 0.26337957873205553,
 'nDCG@5': 0.24061603523156594,
 'nDCG@10': 0.30392706698789657}

In [7]:
scored_df = pd.DataFrame(flat_rows)

metrics_path = OUTPUT_DIR / "metrics.json"
prediction_path = OUTPUT_DIR / "prediction.txt"
scored_path = OUTPUT_DIR / "valid_scored.csv"
runlog_path = OUTPUT_DIR / "run.log"

pd.Series(metrics).to_json(metrics_path, indent=2)
scored_df.to_csv(scored_path, index=False)

with open(prediction_path, "w", encoding="utf-8") as f:
    for imp_id, group in scored_df.groupby("impression_id"):
        ranked = group.sort_values("score", ascending=False)
        ranked_ids = " ".join(ranked["candidate_news_id"].astype(str).tolist())
        f.write(f"{imp_id}\t{ranked_ids}\n")

with open(runlog_path, "w", encoding="utf-8") as f:
    f.write("method=itemcf\n")
    f.write("similarity=jaccard\n")
    f.write("top_k=20\n")
    for key, value in metrics.items():
        f.write(f"{key}={value:.6f}\n")

print("=== Validation Results ===")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

print("Saved:")
print("-", metrics_path)
print("-", prediction_path)
print("-", scored_path)
print("-", runlog_path)


=== Validation Results ===
AUC: 0.5212
MRR: 0.2634
nDCG@5: 0.2406
nDCG@10: 0.3039
Saved:
- /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/round1_baselines/itemcf_output/metrics.json
- /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/round1_baselines/itemcf_output/prediction.txt
- /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/round1_baselines/itemcf_output/valid_scored.csv
- /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/round1_baselines/itemcf_output/run.log
